In [0]:
# set whether normal process or being run by Elsevier supporters
permissions = 'fulldata' # default or fulldata
# set the project name
project_name = 'proj_050_teamscience'  # This is the folder name that you work in, e.g. 'test_project_1'

In [0]:
#Alick: I'll switch to the latest snapshot (v008) - wording there is confusing! 
#%run /Snippets/header_latest

In [0]:
%run /Snippets/header_008

## Python Libraries and Functions

In [0]:
# libraries 
from pyspark.sql.functions import col, count, explode, array, mean, stddev, filter, percentile_approx, arrays_zip, size, array_distinct, sum, reverse, collect_list, array_sort, posexplode, max, arrays_overlap, lit, when, cast
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import Window
from pyspark.ml.feature import StringIndexer

# make a DF to store everything
model_output_df = pd.DataFrame(columns = ["model", "coef", "se", "t", "p", "r2"])
descriptives_output_df = pd.DataFrame(columns = ["model", "subjareas", "mean", "sd", "sample", "median", "iqr_25", "iqr_75"])

# model summary print out 
def modelsummary(model, hyp_model, df):
    import numpy as np
    print ("Note: the last rows are the information for Intercept")
    print ("##","-------------------------------------------------")
    print ("##","  Estimate   |   Std.Error | t Values  |  P-value")
    coef = np.append(list(model.coefficients),model.intercept)
    Summary=model.summary

    for i in range(len(Summary.pValues)):
        print ("##",'{:10.6f}'.format(coef[i]),\
        '{:10.6f}'.format(Summary.coefficientStandardErrors[i]),\
        '{:8.3f}'.format(Summary.tValues[i]),\
        '{:10.6f}'.format(Summary.pValues[i]))
        # store the data 
        row = pd.DataFrame({"model": hyp_model, "coef": coef[i], "se": Summary.coefficientStandardErrors[i], 
               "t": Summary.tValues[i], "p": Summary.pValues[i], "r2": Summary.r2}, index = [0])
        df = pd.concat([df, row], ignore_index = True, axis = 0)

    print ("##",'---')
    print ("##","Mean squared error: % .6f" \
           % Summary.meanSquaredError, ", RMSE: % .6f" \
           % Summary.rootMeanSquaredError )
    print ("##","Multiple R-squared: %f" % Summary.r2, ", \
            Total iterations: %i"% Summary.totalIterations)
    return(df)

## Research Question 1a

### Descriptive Statistics

We will first calculate the mean, median, and interquartile range of team size by discipline. We will not explicitly compare discipline intercepts in a post hoc format with any form of test. Instead we will use visualization techniques to display the distribution and their interquartile range for readers to examine (i.e., a forest plot).

In [0]:
# load the ANI dataframe which contains author information 
df_ani = spark.read.format("parquet").load(basePath+tablename_ani)
df_ani.createOrReplaceTempView("df_ani")
df_ani.printSchema()
df_ani_smaller = df_ani.filter(df_ani.Year > 1969)\
  .filter(df_ani.publication_type == "ar")\
  .sample(False, .25, 8675309) 

df_ani_smaller.count()

# display(df_ani)

root
 |-- Eid: long (nullable = true)
 |-- Year: integer (nullable = true)
 |-- title: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- title: string (nullable = true)
 |    |    |-- lang: string (nullable = true)
 |    |    |-- original: string (nullable = true)
 |-- ASJC: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- subjareas: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- Af: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- affiliation_address_part: string (nullable = true)
 |    |    |-- affiliation_city: string (nullable = true)
 |    |    |-- affiliation_city_group: string (nullable = true)
 |    |    |-- affiliation_country: string (nullable = true)
 |    |    |-- affiliation_ids: array (nullable = true)
 |    |    |    |-- element: long (containsNull = true)
 |    |    |-- affiliation_organization: array (nullable = true)
 |    |    |    |-- 

In [0]:
df_hyp1 = df_ani_smaller.select("Eid", "Year", "Au", "ASJC", "CitationCount")\
  .withColumn("Au", explode("Au"))\
  .withColumn("ASJC", explode("ASJC"))

df_hyp1 = df_hyp1\
  .withColumn('subjareas',
              when(df_hyp1.ASJC.between(1100, 1199), "Life Sciences")\
  .when(df_hyp1.ASJC.between(1200, 1299), "Social Sciences")\
  .when(df_hyp1.ASJC.between(1300, 1399), "Life Sciences")\
  .when(df_hyp1.ASJC.between(1400, 1499), "Social Sciences")\
  .when(df_hyp1.ASJC.between(1500, 1799), "Physical Sciences")\
  .when(df_hyp1.ASJC.between(1800, 1899), "Social Sciences")\
  .when(df_hyp1.ASJC.between(1900, 1999), "Physical Sciences")\
  .when(df_hyp1.ASJC.between(2000, 2099), "Social Sciences")\
  .when(df_hyp1.ASJC.between(2100, 2399), "Physical Sciences")\
  .when(df_hyp1.ASJC.between(2400, 2499), "Life Sciences")\
  .when(df_hyp1.ASJC.between(2500, 2699), "Physical Sciences")\
  .when(df_hyp1.ASJC.between(2700, 2799), "Health Sciences")\
  .when(df_hyp1.ASJC.between(2800, 2899), "Life Sciences")\
  .when(df_hyp1.ASJC.between(2900, 2999), "Health Sciences")\
  .when(df_hyp1.ASJC.between(3000, 3099), "Life Sciences")\
  .when(df_hyp1.ASJC.between(3100, 3199), "Physical Sciences")\
  .when(df_hyp1.ASJC.between(3200, 3399), "Social Sciences")\
  .when(df_hyp1.ASJC.between(3400, 3699), "Health Sciences")\
  .otherwise("Multidisciplinary")
  ).groupBy("Eid", "Year", "subjareas", "CitationCount")\
  .agg(count("Au").alias("team_size"))

In [0]:
summary_df = df_hyp1.select("subjareas", "team_size")\
  .groupBy("subjareas")\
  .agg(mean("team_size").alias("mean"), stddev("team_size").alias("sd"), count("team_size").alias("sample"), percentile_approx("team_size", 0.5, 10000).alias("median"), percentile_approx("team_size", 0.25, 10000).alias("iqr_25"), percentile_approx("team_size", 0.75, 10000).alias("iqr_75"))\
  .toPandas()
summary_df["model"] = "hyp1a"
descriptives_output_df = pd.concat([descriptives_output_df, summary_df], ignore_index = True, axis = 0)

# descriptives_output_df

### Overall Summary

In order to determine if team sizes change overall across time, we will use a linear model of team size predicted by year of publication in which each article is only represented once, regardless of discipline. We will examine if the confidence interval of the slope parameter is different from zero to determine if the predictor was significant.

In [0]:
# 4.19s on default #54.96 seconds on fulldata 
# get the total number of authors by year 
df_hyp1_overall = df_ani_smaller.select("Eid", "Year", "Au", "CitationCount", explode("Au").alias("author"))\
  .groupBy("Eid", "Year", "CitationCount")\
  .agg(count("author").alias("team_size"))

# display(df_hyp1_overall)

# team size ~ year of publication on the entire dataset 
# the "features" or X must be in this feature format
vectorAssembler = VectorAssembler(inputCols = ['Year'], outputCol = "pub_year")
df_hyp1_overall = vectorAssembler.setHandleInvalid("skip").transform(df_hyp1_overall)

# put in the features X and prediction Y
lr = LinearRegression(featuresCol = 'pub_year', labelCol='team_size')
lr_model = lr.fit(df_hyp1_overall)
model_output_df = modelsummary(lr_model, "hyp1a", model_output_df)

# model_output_df

Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
##   0.075928   0.000321  236.450   0.000000
## -148.041472   0.643774 -229.959   0.000000
## ---
## Mean squared error:  315.727625 , RMSE:  17.768726
## Multiple R-squared: 0.003825 ,             Total iterations: 0


### Summary By Subject Area

Last, we will calculate each discipline's slope and confidence interval using a linear model of team size predicted by year of publication. Again, we will provide these slopes in a visualization for readers to examine with no official post hoc comparison. While a multilevel model would allow us to embed all disciplines within one model, no *pyspark* multilevel model implementation exists to model data at this scale.

In [0]:


for AREA in descriptives_output_df.subjareas.unique():
  if AREA == 'undefined': continue
  print(AREA)
  print("\n-------\n")
  temp_df = df_hyp1.select("subjareas", "team_size", "Year")\
    .filter(df_hyp1.subjareas == AREA)
  
  vectorAssembler = VectorAssembler(inputCols = ['Year'], outputCol = "pub_year")
  temp_df = vectorAssembler.setHandleInvalid("skip").transform(temp_df)
  lr = LinearRegression(featuresCol = 'pub_year', labelCol='team_size')
  lr_model = lr.fit(temp_df)
  model_output_df = modelsummary(lr_model, ("hyp1a_"+AREA), model_output_df)
  
# model_output_df

Health Sciences

-------

Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
##   0.183990   0.009037   20.361   0.000000
## -359.829970  18.155336  -19.820   0.000000
## ---
## Mean squared error:  285.209701 , RMSE:  16.888153
## Multiple R-squared: 0.010320 ,             Total iterations: 0
Physical Sciences

-------

Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
##   0.265573   0.026587    9.989   0.000000
## -521.409270  53.454498   -9.754   0.000000
## ---
## Mean squared error:  4114.805612 , RMSE:  64.146751
## Multiple R-squared: 0.001462 ,             Total iterations: 0
Multidisciplinary

-------

Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
#

## Research Question 1b

### Descriptive Statistics

For this research question, we will examine the impact of team size on article publication metrics for impact. Our hypothesis is that team size impacts article metrics (non-directional). We will run these analyses once for each dependent variable of citation count and weighted citation five year count. We will calculate the mean, median, and interquartile range of each publication metric by discipline. We will visualize these values instead of comparing them directly.

In [0]:
# grab Citations_5y from df_fwci
df_fwci = spark.read.format("parquet").load(fwci_parquet_path)
df_fwci.createOrReplaceTempView("df_fwci")

# display(df_fwci)

# merge this information with tables from above 
df_hyp1_overall = df_hyp1_overall.join(df_fwci.select("Eid", "Citations_5y"), "Eid", "left")

df_hyp1 = df_hyp1.join(df_fwci.select("Eid", "Citations_5y"), "Eid", "left")

In [0]:
# summarize citation count
summary_df = df_hyp1.select("subjareas", "CitationCount")\
  .groupBy("subjareas")\
  .agg(mean("CitationCount").alias("mean"), 
       stddev("CitationCount").alias("sd"), 
       count("CitationCount").alias("sample"), 
       percentile_approx("CitationCount", 0.5, 10000).alias("median"),
       percentile_approx("CitationCount", 0.25, 10000).alias("iqr_25"), 
       percentile_approx("CitationCount", 0.75, 10000).alias("iqr_75"))\
  .toPandas()
summary_df["model"] = "hyp1b_CitationCount"
descriptives_output_df = pd.concat([descriptives_output_df, summary_df], ignore_index = True, axis = 0)

# summarize citation 5y, exclude last 5 years in graph
summary_df = df_hyp1.select("subjareas", "Citations_5y")\
  .groupBy("subjareas")\
  .agg(mean("Citations_5y").alias("mean"), 
       stddev("Citations_5y").alias("sd"), 
       count("Citations_5y").alias("sample"), 
       percentile_approx("Citations_5y", 0.5, 10000).alias("median"),
       percentile_approx("Citations_5y", 0.25, 10000).alias("iqr_25"), 
       percentile_approx("Citations_5y", 0.75, 10000).alias("iqr_75"))\
  .toPandas()
summary_df["model"] = "hyp1b_Citation5y"
descriptives_output_df = pd.concat([descriptives_output_df, summary_df], ignore_index = True, axis = 0)

### Overall Summary 
To determine the impact of team size on publication metrics, we will use a linear model of publication metrics predicted by time (covariate) and team size. We will include time as a covariate, as we expect that older articles will potentially have more or different metrics because they have been around longer to cite. We will examine if the confidence interval of the team size slope parameter is different from zero.

In [0]:
# citation count ~ year of publication + team_size
# the "features" or X must be in this feature format
df_hyp1_overall = df_hyp1_overall.drop("pub_year")
vectorAssembler = VectorAssembler(inputCols = ['Year', 'team_size'], outputCol = "pub_year")
df_hyp1_overall = vectorAssembler.setHandleInvalid("skip").transform(df_hyp1_overall)

# put in the features X and prediction Y
lr = LinearRegression(featuresCol = 'pub_year', labelCol = 'CitationCount')
lr_model = lr.fit(df_hyp1_overall)
model_output_df = modelsummary(lr_model, "hyp1b_CitationCount", model_output_df)

# model_output_df

Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
## -13.922517   0.212294  -65.581   0.000000
##   0.581451   0.061245    9.494   0.000000
## 28122.322635 426.607220   65.921   0.000000
## ---
## Mean squared error:  583112.970305 , RMSE:  763.618341
## Multiple R-squared: 0.029424 ,             Total iterations: 0


In [0]:
# citation_5y ~ year of publication + team_size
# the "features" or X must be in this feature format
df_hyp1_overall_sub = df_hyp1_overall.drop("pub_year")\
  .filter(df_hyp1_overall.Year < 2017).where(col("Citations_5y").isNotNull())
vectorAssembler = VectorAssembler(inputCols = ['Year', 'team_size'], outputCol = "pub_year")
df_hyp1_overall = vectorAssembler.setHandleInvalid("skip").transform(df_hyp1_overall_sub)

# put in the features X and prediction Y
lr = LinearRegression(featuresCol = 'pub_year', labelCol = 'Citations_5y')
lr_model = lr.fit(df_hyp1_overall)
model_output_df = modelsummary(lr_model, "hyp1b_Citation5y", model_output_df)

# model_output_df

Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
##   0.622689   0.050632   12.298   0.000000
##   0.379534   0.011883   31.939   0.000000
## -1208.284562 101.575004  -11.895   0.000000
## ---
## Mean squared error:  21314.166674 , RMSE:  145.993721
## Multiple R-squared: 0.011203 ,             Total iterations: 0


### Summary By Subject Area
Last, we will calculate the slope for team size for each discipline separately. We will visualize these differences and the confidence intervals.

In [0]:
for AREA in descriptives_output_df.subjareas.unique():
  if AREA == 'undefined': continue
  print(AREA)
  print("\n-------\n")
  temp_df = df_hyp1.select("subjareas", "team_size", "Year", "CitationCount", "Citations_5y")\
    .filter(df_hyp1.subjareas == AREA)
  
  vectorAssembler = VectorAssembler(inputCols = ['Year', 'team_size'], outputCol = "pub_year")
  temp_df = vectorAssembler.setHandleInvalid("skip").transform(temp_df)
  lr = LinearRegression(featuresCol = 'pub_year', labelCol='CitationCount')
  lr_model = lr.fit(temp_df)
  model_output_df = modelsummary(lr_model, ("hyp1b_CitationCount_"+AREA), model_output_df)
  
  lr = LinearRegression(featuresCol = 'pub_year', labelCol='Citations_5y')
  lr_model = lr.fit(temp_df.filter(temp_df.Year < 2017).where(col("Citations_5y").isNotNull()))
  model_output_df = modelsummary(lr_model, ("hyp1b_Citation5y_"+AREA), model_output_df)
  
# model_output_df

Health Sciences

-------

Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
## -10.314310   0.272629  -37.833   0.000000
##   1.884351   0.150525   12.519   0.000000
## 20832.313046 547.587849   38.044   0.000000
## ---
## Mean squared error:  256910.856621 , RMSE:  506.863746
## Multiple R-squared: 0.036530 ,             Total iterations: 0
Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
##   0.398839   0.126252    3.159   0.001584
##   1.817312   0.082325   22.075   0.000000
## -773.215033 253.163072   -3.054   0.002258
## ---
## Mean squared error:  36399.360624 , RMSE:  190.786165
## Multiple R-squared: -0.000324 ,             Total iterations: 0
Physical Sciences

-------

Note: the last rows are the information for Intercept
## ----------------------------------

# Output Tables

Output tables for writing manuscript in R markdown.

In [0]:
# write out results section until I can get download link
pd.set_option('display.max_rows', None)
model_output_df

,model,coef,se,t,p,r2
0,hyp1a,0.101687,0.009138,11.127818,0.0,0.000861
1,hyp1a,-198.50472,18.363503,-10.809741,0.0,0.000861
2,hyp1a_Health Sciences,0.18399,0.009037,20.360572,0.0,0.01032
3,hyp1a_Health Sciences,-359.82997,18.155336,-19.819516,0.0,0.01032
4,hyp1a_Physical Sciences,0.265573,0.026587,9.988784,0.0,0.001462
5,hyp1a_Physical Sciences,-521.40927,53.454498,-9.754264,0.0,0.001462
6,hyp1a_Multidisciplinary,0.11362,0.056635,2.006186,0.044885,0.000731
7,hyp1a_Multidisciplinary,-219.613206,113.770369,-1.93032,0.053619,0.000731
8,hyp1a_Social Sciences,0.060509,0.005188,11.663092,0.0,0.008043
9,hyp1a_Social Sciences,-116.552976,10.42511,-11.180023,0.0,0.008043


In [0]:
descriptives_output_df

,model,subjareas,mean,sd,sample,median,iqr_25,iqr_75
0,hyp1a,Health Sciences,9.819035,16.976186,39759,7,4,12
1,hyp1a,Physical Sciences,12.530522,64.194176,68131,8,4,15
2,hyp1a,Multidisciplinary,8.628452,42.429613,5504,6,4,9
3,hyp1a,Social Sciences,5.034569,6.673358,16778,4,2,6
4,hyp1a,Life Sciences,10.733374,12.479251,48945,8,4,13
5,hyp1b_CitationCount,Health Sciences,128.641289,516.389834,39759,34,9,100
6,hyp1b_CitationCount,Physical Sciences,127.04961,763.368957,68131,27,7,85
7,hyp1b_CitationCount,Multidisciplinary,392.900073,1271.384244,5504,83,15,359
8,hyp1b_CitationCount,Social Sciences,164.664382,740.622985,16778,26,5,102
9,hyp1b_CitationCount,Life Sciences,137.362059,754.269326,48945,37,11,102


PROJECT DETAILS:
Project identifier: proj_050_teamscience
Cluster data access level: fulldata


EXECUTION DETAILS:
Spark version: 11.3.x-aarch64-scala2.12, 
Cluster Node Type: r6gd.2xlarge, 
Driver Node Type: r6gd.2xlarge, 
Worker Node Type: r6gd.2xlarge


DATA SNAPSHOT DETAILS:
This notebook uses the latest snapshot date for this version: "v008.20230509104116/", (09 May 2023)

 The following are the Scopus snapshot dates available for datasets at v008:
['v008.20220314092239/', 'v008.20220906063204/', 'v008.20221021151538/', 'v008.20230509104116/']
(If you want to use a different snapshot, in the first cell of your notebook, set the variable `custom_snapshot_date` to one of the options listed above (string data type))



DATA SAMPLE DETAILS:
Setting paths to run on full datasets (100% Scopus publication volume)


In [0]:
model_output_df.to_csv('/dbfs/Projects/proj_050_teamsci/model_output_200723_part1.csv')
descriptives_output_df.to_csv('/dbfs/Projects/proj_050_teamsci/descriptives_output_df_200723_part1.csv')